In [7]:
!pip install transformers torch datasets pyarrow boto3 pandas tf-keras --quiet

In [1]:
import boto3
import pandas as pd
from io import BytesIO

BUCKET = "candrews-sentiment-pipeline"
s3 = boto3.client("s3")

obj = s3.get_object(Bucket=BUCKET, Key="raw/imdb_reviews.parquet")
df = pd.read_parquet(BytesIO(obj["Body"].read()))

print(f"Loaded {len(df)} rows from s3")
df.head()

Loaded 500 rows from s3


,text,label
0,<br /><br />When I unsuspectedly rented A Thou...,1
1,This is the latest entry in the long series of...,1
2,This movie was so frustrating. Everything seem...,0
3,"I was truly and wonderfully surprised at ""O' B...",1
4,This movie spends most of its time preaching t...,0


In [2]:
from transformers import pipeline
# distilbert-base-uncased-finetuned-sst-2-english is:
# - DistilBERT: a smaller, faster version of BERT (40% smaller, 60% faster, 97% accuracy)
# - uncased: treats uppercase and lowercase as the same
# - finetuned-sst-2: trained specifically on sentiment classification data
# - english: English language only
classifer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1)

# Convert reviews to a list
texts = df["text"].tolist()

# truncation=True handles reviews longer than 512 tokens (BERT's hard limit)
results = classifer(texts, truncation=True, max_length=512)

# Add predictions back to the dataframe
df['predicted_label'] = [r['label'] for r in results]
df['confidence'] = [round(r['score'], 4) for r in results]

df.head()

Device set to use cpu


,text,label,predicted_label,confidence
0,<br /><br />When I unsuspectedly rented A Thou...,1,POSITIVE,0.9989
1,This is the latest entry in the long series of...,1,POSITIVE,0.9970
2,This movie was so frustrating. Everything seem...,0,NEGATIVE,0.9972
3,"I was truly and wonderfully surprised at ""O' B...",1,NEGATIVE,0.6492
4,This movie spends most of its time preaching t...,0,NEGATIVE,0.9985


In [3]:
# IMDB labels: 0 = negative, 1 = positive
# Model returns: "NEGATIVE" or "POSITIVE"
# Map them to match so we can compare
label_map = {"NEGATIVE": 0, "POSITIVE": 1}
df['predicted_int'] = df['predicted_label'].map(label_map)

accuracy = (df['predicted_int'] == df['label']).mean()
print(f"Accuracy on sample: {accuracy:.1%}")

Accuracy on sample: 89.0%


In [4]:
from io import BytesIO

# Write to an in-memory buffer instead of disk
# Lambda has very limited disk space so this habit transfers well
buffer = BytesIO()
df.to_parquet(buffer, index=False)

s3.put_object(
    Bucket=BUCKET,
    Key="results/imdb_results.parquet",
    Body=buffer.getvalue()
)

print(f"Results saved to s3://{BUCKET}/results/imdb_results.parquet")

Results saved to s3://candrews-sentiment-pipeline/results/imdb_results.parquet
